# NEXUS OptiML 2026 – Product Return Prediction (ROC-AUC)

**Key insight from EDA:** the effect of `promo_channel_code` *decays over time* (P5 ≈ 44 % returns vs P0 ≈ 9 % in 2023, but all codes ≈ 18 % by Dec-2024). A model that learns it fails on the future test months, so the notebook
1. diagnoses the drift (graphs),
2. cleans dirty labels / duplicate customers and fills nulls **logically from related tables**,
3. engineers stable multi-table features + out-of-fold target encoding,
4. validates on a **time-based** split (like the real test),
5. neutralises the promo drift (drop vs. learned offset – picked by CV),
6. blends LightGBM + XGBoost + CatBoost + a standardised logistic regression (rank average, weights tuned on the latest fold).

Set `SAMPLE_FRAC < 1` for a quick smoke test.

In [ ]:
import os, gc, re, glob, time, difflib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import sparse
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
warnings.filterwarnings('ignore'); sns.set_style('whitegrid')
import lightgbm as lgb
try:    import xgboost as xgb;                       HAS_XGB = True
except ImportError: HAS_XGB = False
try:    from catboost import CatBoostClassifier, Pool; HAS_CB = True
except ImportError: HAS_CB = False

# ---------------- configuration ----------------
cands = [os.environ.get('DATA_DIR', '.')] + glob.glob('/kaggle/input/*/') + glob.glob('/kaggle/input/*/*/') + ['/mnt/user-data/uploads']
DATA_DIR    = next(d for d in cands if os.path.exists(os.path.join(d, 'train.csv')))
SAMPLE_FRAC = float(os.environ.get('SAMPLE_FRAC', 1.0))   # <1 => hash-sample orders for a fast run
SEED        = 42
USE_XGB     = HAS_XGB and os.environ.get('USE_XGB', '1') == '1'
USE_CB      = HAS_CB  and os.environ.get('USE_CB',  '1') == '1'
USE_LR      = os.environ.get('USE_LR', '1') == '1'
LGB_ROUNDS, XGB_ROUNDS, CB_ROUNDS = 4000, 3000, 1500
# time-based validation folds (val_start, val_end); each fold trains on everything BEFORE val_start
FOLDS = [('2024-08-01', '2024-11-01'), ('2024-11-01', '2025-01-02')]
print('data dir:', DATA_DIR, '| sample:', SAMPLE_FRAC, '| xgb/cb/lr:', USE_XGB, USE_CB, USE_LR)

## 1. Load data

In [ ]:
tr  = pd.read_csv(f'{DATA_DIR}/train.csv');   te  = pd.read_csv(f'{DATA_DIR}/test.csv')
prd = pd.read_csv(f'{DATA_DIR}/products.csv'); st = pd.read_csv(f'{DATA_DIR}/stores.csv')
cu  = pd.read_csv(f'{DATA_DIR}/customers.csv'); cal = pd.read_csv(f'{DATA_DIR}/calendar.csv')
if SAMPLE_FRAC < 1:
    h = lambda d: (pd.util.hash_array(d.order_id.values) % 1000) < SAMPLE_FRAC * 1000
    tr, te = tr[h(tr)].reset_index(drop=True), te[h(te)].reset_index(drop=True)
print(tr.shape, te.shape, prd.shape, st.shape, cu.shape, cal.shape)
print('overall return rate: %.4f' % tr.returned.mean())

## 2. EDA – nulls, drift, and why `promo_channel_code` must be handled

In [ ]:
# --- null map of every raw table (what has to be repaired) ---
fig, ax = plt.subplots(1, 5, figsize=(22, 4))
for a, (n, d) in zip(ax, [('train', tr), ('customers', cu), ('products', prd), ('stores', st), ('calendar', cal)]):
    s = d.isnull().mean().mul(100); s = s[s > 0]
    (s.sort_values().plot.barh(ax=a, color='crimson') if len(s) else a.text(.3, .5, 'no nulls'))
    a.set_title(f'{n}: % null'); 
plt.tight_layout(); plt.show()

In [ ]:
# --- concept drift: return rate by month x promo code ---
tr['_m'] = pd.to_datetime(tr.order_ts).dt.to_period('M').astype(str)
piv = tr[tr._m.between('2023-01', '2024-12')].pivot_table(index='_m', columns='promo_channel_code', values='returned', aggfunc='mean')
fig, ax = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(piv, annot=True, fmt='.2f', cmap='RdYlBu_r', ax=ax[0]); ax[0].set_title('Return rate: month x promo code (effect collapses in 2024)')
piv.plot(ax=ax[1], marker='o'); ax[1].set_title('Return rate by promo code over time'); ax[1].tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()
print('Spread P5-P0 by quarter:'); print((piv['P5'] - piv['P0']).groupby(lambda m: m[:4] + 'Q' + str((int(m[5:]) - 1) // 3 + 1)).mean().round(3).to_dict())

In [ ]:
# --- other categorical effects (stable over time -> safe to use) ---
fig, ax = plt.subplots(2, 3, figsize=(20, 9))
for a, c in zip(ax.ravel(), ['channel', 'payment_method', 'return_window_days', 'device_type', 'ship_mode', 'tz_hint']):
    tr.fillna({c: 'NULL'}).groupby(c).returned.mean().sort_values().plot.bar(ax=a, color='steelblue'); a.set_title(f'return rate by {c}'); a.axhline(tr.returned.mean(), color='r', ls='--')
plt.tight_layout(); plt.show()

## 3. Cleaning, logical null-filling and multi-table joins

| Problem | Fix |
|---|---|
| product `category` has ~60 spellings of 12 classes | fuzzy-match to the clean class list |
| customer `region` has case/space/typo variants | fuzzy-match to the clean store-region list |
| 3 029 duplicate customer rows | keep latest record, fill nulls from older rows, keep `n_records`, `tier_changed` |
| `age_band` null | same customer's other record → else mode of (customer_type × loyalty_tier) |
| `size_sqm` null (77 % of stores) | structural: physical formats → format median, online/marketplace sellers → 0 (+flag) |
| `weight_kg` null | GBDT regression from category, dimensions, price, packaging |
| `freight_cost` null | GBDT regression from channel, ship mode, quantity, weight, regions … |
| `discount` null | GBDT regression from channel, category, customer type, price … |
| `session_clicks` null | phone/POS have no web session → 0; others regression on device/channel/hour |
| `payment_method` null | customer's usual method → else channel's most common |
| `ship_mode` null | store channel → `in_store`, else mode |
| `tz_hint` null | explicit `unknown` level |
| calendar | week-over-week changes + de-trended temperature (raw CPI/unemployment levels do not extrapolate to test) |

In [ ]:
key = lambda s: re.sub(r'[^a-z]', '', str(s).lower())
def canon_map(values, canon, cutoff=0.75):
    ck = {key(c): c for c in canon}; out = {}
    for v in pd.unique(values):
        k = key(v)
        if k in ck: out[v] = ck[k]
        else:
            r = difflib.get_close_matches(k, list(ck), 1, cutoff); out[v] = ck[r[0]] if r else v
    return out

def model_impute(df, target, feats, cats, n_fit=150000, seed=0, log=False):
    """Fill NaNs of df[target] with a GBDT regressor trained on the observed rows (relations, not random values)."""
    miss = df[target].isnull()
    if not miss.any(): return
    X = df[feats].copy()
    for c in cats: X[c] = X[c].astype('category').cat.codes
    obs = df.index[~miss]; obs = pd.Index(np.random.RandomState(seed).choice(obs, min(n_fit, len(obs)), replace=False))
    y = df.loc[obs, target]; y = np.log1p(y) if log else y
    m = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.1, categorical_features=[X.columns.get_loc(c) for c in cats] or None, random_state=seed)
    m.fit(X.loc[obs], y); p = m.predict(X.loc[miss]); df.loc[miss, target] = np.expm1(p) if log else p

def clean_tables(pr, st, cu, cal):
    # products
    top = [c for c, n in pr.category.value_counts().items() if n >= 300 and c == c.lower() and ' ' not in c and c == c.strip()]
    pr['category'] = pr.category.map(canon_map(pr.category.unique(), top))
    pr['vol'] = pr.length_cm * pr.width_cm * pr.height_cm
    pr['weight_missing'] = pr.weight_kg.isnull().astype('int8')
    model_impute(pr, 'weight_kg', ['category', 'length_cm', 'width_cm', 'height_cm', 'list_price', 'packaging_type', 'review_count'], ['category', 'packaging_type'], log=True)
    pr['density'] = pr.weight_kg / (pr.vol + 1e-3)
    pr['price_to_cat_median'] = pr.list_price / pr.groupby('category').list_price.transform('median')
    pr['launch_date'] = pd.to_datetime(pr.launch_date)
    # stores
    st['region'] = st.region.map(canon_map(st.region.unique(), list(st.region.unique())))
    st['size_missing'] = st.size_sqm.isnull().astype('int8')
    st['size_sqm'] = st.size_sqm.fillna(st.groupby('format').size_sqm.transform('median')).fillna(0)
    st['opened_date'] = pd.to_datetime(st.opened_date); st['is_physical'] = st.channel.isin(['store', 'catalog']).astype('int8')
    # customers: canonical region, merge duplicates (latest wins, nulls filled from older records)
    cu['region'] = cu.region.map(canon_map(cu.region.unique(), list(st.region.unique())))
    cu['record_updated_at'] = pd.to_datetime(cu.record_updated_at); cu = cu.sort_values('record_updated_at')
    cu['n_records'] = cu.groupby('customer_id').customer_id.transform('size').astype('int8')
    first_tier = cu.groupby('customer_id').loyalty_tier.first().rename('tier_first')
    cu = cu.groupby('customer_id', as_index=False).last().merge(first_tier, on='customer_id')
    cu['tier_changed'] = (cu.tier_first != cu.loyalty_tier).astype('int8'); cu['age_missing'] = cu.age_band.isnull().astype('int8')
    mode = cu.groupby(['customer_type', 'loyalty_tier']).age_band.transform(lambda s: s.mode().iat[0] if s.notnull().any() else np.nan)
    cu['age_band'] = cu.age_band.fillna(mode); cu['signup_date'] = pd.to_datetime(cu.signup_date)
    # calendar: stationary versions only
    cal['week_start'] = pd.to_datetime(cal.week_start); cal = cal.sort_values('week_start')
    for c in ['cpi', 'fuel_price', 'unemployment']: cal[c + '_chg'] = cal[c].diff().fillna(0)
    cal['temp_dev'] = cal.temperature_f - cal.temperature_f.rolling(52, min_periods=1, center=True).mean()
    return pr, st, cu, cal

In [ ]:
def build(tr, te, pr, st, cu, cal):
    pr, st, cu, cal = clean_tables(pr, st, cu, cal)
    tr = tr.drop(columns=['_m'], errors='ignore'); tr['is_train'] = 1; te['is_train'] = 0; te['returned'] = np.nan
    df = pd.concat([tr, te], ignore_index=True)
    df['ts'] = pd.to_datetime(df.order_ts); df['week_start'] = df.ts.dt.normalize() - pd.to_timedelta(df.ts.dt.dayofweek, unit='D')
    df = df.merge(cu.drop(columns=['record_updated_at']).rename(columns={'region': 'c_region', 'customer_type': 'c_type'}), on='customer_id', how='left')
    df = df.merge(pr.rename(columns={'category': 'cat'}), on='product_id', how='left')
    df = df.merge(st.rename(columns={'channel': 's_channel', 'region': 's_region', 'format': 's_format'}), on='store_id', how='left')
    df = df.merge(cal[['week_start', 'holiday_flag', 'temperature_f', 'fuel_surcharge', 'cpi_chg', 'fuel_price_chg', 'unemployment_chg', 'temp_dev']], on='week_start', how='left')
    # ---- logical null filling of transaction columns
    for c, f in [('discount', 'discount_missing'), ('freight_cost', 'freight_missing'), ('session_clicks', 'clicks_missing')]: df[f] = df[c].isnull().astype('int8')
    df['lv'] = df.unit_price * df.quantity; df['lq'] = np.log1p(df.quantity)
    df['hour'] = df.ts.dt.hour; df['dow'] = df.ts.dt.dayofweek; df['month'] = df.ts.dt.month
    df['date_only'] = (df.ts.dt.minute == 1).astype('int8')            # date-only placeholder timestamps
    df['ship_mode'] = df.ship_mode.fillna(df.channel.map({'store': 'in_store'})).fillna(df.ship_mode.mode().iat[0])
    ch_pm = df.groupby('channel').payment_method.agg(lambda s: s.mode().iat[0])
    cust_pm = df.dropna(subset=['payment_method']).groupby('customer_id').payment_method.agg(lambda s: s.mode().iat[0])
    df['payment_method'] = df.payment_method.fillna(df.customer_id.map(cust_pm)).fillna(df.channel.map(ch_pm))
    df['tz_hint'] = df.tz_hint.fillna('unknown')
    nosess = df.device_type.isin(['phone', 'pos']); df.loc[nosess, 'session_clicks'] = df.loc[nosess, 'session_clicks'].fillna(0)
    model_impute(df, 'session_clicks', ['device_type', 'channel', 'items_in_order', 'hour', 'dow'], ['device_type', 'channel'], seed=1)
    model_impute(df, 'freight_cost', ['channel', 'ship_mode', 'lq', 'weight_kg', 's_region', 'c_region', 'items_in_order', 'unit_price'], ['channel', 'ship_mode', 's_region', 'c_region'], seed=2, log=True)
    model_impute(df, 'discount', ['channel', 'cat', 'c_type', 'loyalty_tier', 'lq', 'unit_price', 'list_price', 'return_window_days'], ['channel', 'cat', 'c_type', 'loyalty_tier'], seed=3)
    # ---- money / ratio features
    df['price_ratio'] = df.unit_price / df.list_price; df['log_price_ratio'] = np.log(df.price_ratio)
    df['ov_ratio'] = df.order_value / (df.lv + 1e-4); df['ov_diff'] = df.order_value - df.lv; df['net_val'] = df.lv * (1 - df.discount)
    df['freight_per_unit'] = df.freight_cost / df.quantity; df['freight_share'] = df.freight_cost / (df.order_value + 1)
    df['log_lv'] = np.log1p(df.lv); df['log_price'] = np.log1p(df.unit_price); df['log_ov'] = np.log1p(df.order_value)
    df['window_x_price'] = df.return_window_days * df.log_price
    df['has_coupon'] = df.coupon_code.notnull().astype('int8'); df['clicks_per_item'] = df.session_clicks / df.items_in_order
    df['line_pos'] = df.line_no / df.items_in_order
    df['cross_region'] = (df.c_region != df.s_region).astype('int8')
    df['cust_country'] = df.c_region.str.split('-').str[0]; df['store_country'] = df.s_region.str.split('-').str[0]
    df['days_since_signup'] = (df.ts - df.signup_date).dt.days; df['prod_age'] = (df.ts - df.launch_date).dt.days; df['store_age'] = (df.ts - df.opened_date).dt.days
    # ---- order context (orders never straddle train/test)
    g = df.groupby('order_id')
    df['o_lines'] = g.order_line_id.transform('size'); df['o_qty'] = g.quantity.transform('sum'); df['o_nprod'] = g.product_id.transform('nunique')
    df['o_lv'] = g.lv.transform('sum'); df['o_cover'] = df.o_lines / df.items_in_order
    df['qty_share'] = df.quantity / df.o_qty; df['val_share'] = df.lv / df.o_lv
    df['price_rank'] = g.unit_price.rank(pct=True); df['o_ncat'] = g.cat.transform('nunique'); df['o_maxprice'] = g.unit_price.transform('max')
    # ---- label-free entity statistics (train+test together; no target used)
    for c in ['customer_id', 'product_id', 'store_id', 'supplier_code']: df[c + '_n'] = df.groupby(c).order_line_id.transform('size')
    df['is_mkt'] = (df.s_channel == 'marketplace').astype('int8')
    gc_ = df.groupby('customer_id')
    df['cust_orders'] = gc_.order_id.transform('nunique'); df['cust_mean_lv'] = gc_.log_lv.transform('mean'); df['cust_mean_disc'] = gc_.discount.transform('mean')
    df['cust_n_cat'] = gc_.cat.transform('nunique'); df['cust_mkt_share'] = gc_.is_mkt.transform('mean'); df['cust_mean_ratio'] = gc_.log_price_ratio.transform('mean')
    gp = df.groupby('product_id')
    df['prod_mean_ratio'] = gp.log_price_ratio.transform('mean'); df['prod_mean_qty'] = gp.lq.transform('mean')
    df['prod_n_cust'] = gp.customer_id.transform('nunique'); df['prod_mean_disc'] = gp.discount.transform('mean')
    gs = df.groupby('store_id')
    df['store_mean_lv'] = gs.log_lv.transform('mean'); df['store_mean_disc'] = gs.discount.transform('mean'); df['store_n_prod'] = gs.product_id.transform('nunique')
    df['cust_x_store_n'] = df.groupby(['customer_id', 'store_id']).order_line_id.transform('size')
    df['cust_prev_gap'] = df.sort_values('ts').groupby('customer_id').ts.diff().dt.total_seconds().div(86400).reindex(df.index)
    # ---- categorical -> integer codes
    CATS = ['channel', 'payment_method', 'ship_mode', 'device_type', 'promo_channel_code', 'warehouse_zone', 'picker_shift', 'tz_hint', 'cat', 'c_type', 'age_band',
            'loyalty_tier', 'tier_first', 's_format', 's_channel', 's_region', 'c_region', 'packaging_type', 'color', 'preferred_contact', 'cust_country', 'store_country']
    for c in CATS: df[c] = df[c].astype('category').cat.codes.astype('int16')
    return df.copy(), CATS

t0 = time.time()
df, CATS = build(tr, te, prd, st, cu, cal)
del tr, te, prd, st, cu, cal; gc.collect()
print(df.shape, f'{time.time()-t0:.0f}s')

In [ ]:
ID_COLS = ['order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id', 'order_ts', 'returned', 'is_train', 'ts', 'week_start',
           'signup_date', 'launch_date', 'opened_date', 'coupon_code', 'supplier_code', '_m']
FEATS = [c for c in df.columns if c not in ID_COLS and c != 'promo_channel_code']   # promo is handled separately (drift)
for c in FEATS:
    if df[c].dtype == 'float64': df[c] = df[c].astype('float32')
left = df[FEATS].isnull().sum(); print('remaining nulls in features:', left[left > 0].to_dict(), '(first-order gap is a genuine "no previous order")')
T    = df[df.is_train == 1].reset_index(drop=True); TEST = df[df.is_train == 0].reset_index(drop=True); y_all = T.returned.values.astype(int)
del df; gc.collect(); print('train', T.shape, 'test', TEST.shape, '| features:', len(FEATS))

## 4. Correlations & feature relevance

In [ ]:
num_f = [c for c in FEATS if c not in CATS]
samp = T.sample(min(300000, len(T)), random_state=SEED)
cor_t = samp[num_f].corrwith(samp.returned, method='spearman').dropna()
top = cor_t.abs().sort_values(ascending=False).head(30).index
fig, ax = plt.subplots(1, 2, figsize=(22, 9))
cor_t[top].sort_values().plot.barh(ax=ax[0], color='teal'); ax[0].set_title('Top-30 Spearman correlation with `returned`')
sns.heatmap(samp[list(top[:22])].corr(), cmap='coolwarm', center=0, ax=ax[1], cbar_kws={'shrink': .7}); ax[1].set_title('Correlation heatmap of top features')
plt.tight_layout(); plt.show()

In [ ]:
# monthly return rate (stationary after removing promo) + distribution of key numeric features by target
fig, ax = plt.subplots(1, 3, figsize=(21, 4.5))
T.groupby(T.ts.dt.to_period('M').astype(str)).returned.mean().iloc[1:-1].plot(ax=ax[0], marker='o'); ax[0].set_title('overall return rate by month (stable)'); ax[0].tick_params(axis='x', rotation=90)
for v, a in zip(['log_lv', 'customer_id_n'], ax[1:]):
    sns.kdeplot(data=samp, x=v, hue='returned', common_norm=False, ax=a); a.set_title(v + ' by target')
plt.tight_layout(); plt.show()

## 5. Target encoding, promo-drift offset, models

* **Target encoding** of customer / product / store: smoothed mean, *out-of-fold* (GroupKFold by `order_id`) for training rows, full-train mapping for validation/test rows – no leakage.
* **Promo drift:** `drop` = simply exclude the promo column. `offset` = estimate the (month × promo) effect, feed it as a fixed logit offset (`init_score` / `base_margin` / `baseline`) so trees learn *only* the stable signal; at prediction time no offset is used. Picked automatically on the most recent fold (where the promo effect is already ~0).
* **Standardisation / normalisation:** trees do not need it; the logistic-regression model gets quantile→normal scaling for numerics and one-hot encoding for categoricals.

In [ ]:
TE_COLS = ['customer_id', 'product_id', 'store_id']
def target_encode(A, yA, B, cols=TE_COLS, k=20, n_splits=5):
    prior = yA.mean(); oof = {}; app = {}
    for c in cols: oof[c + '_te'] = np.full(len(A), prior, dtype='float32')
    for a, b in GroupKFold(n_splits).split(A, groups=A.order_id.values):
        for c in cols:
            s = pd.DataFrame({'k': A[c].values[a], 'y': yA[a]}).groupby('k').y.agg(['sum', 'count']); m = (s['sum'] + k * prior) / (s['count'] + k)
            oof[c + '_te'][b] = pd.Series(A[c].values[b]).map(m).fillna(prior).values
    for c in cols:
        s = pd.DataFrame({'k': A[c].values, 'y': yA}).groupby('k').y.agg(['sum', 'count']); m = (s['sum'] + k * prior) / (s['count'] + k)
        app[c + '_te'] = pd.Series(B[c].values).map(m).fillna(prior).values.astype('float32')
    return pd.DataFrame(oof), pd.DataFrame(app)

def promo_offset(A, yA, k=50):
    """logit of the smoothed (month x promo) return rate of the TRAIN part: captures the fading promo effect."""
    d = pd.DataFrame({'m': A.ts.dt.to_period('M').astype(str).values, 'p': A.promo_channel_code.values, 'y': yA})
    d['mm'] = d.groupby('m').y.transform('mean')
    g = d.groupby(['m', 'p']).agg(s=('y', 'sum'), n=('y', 'size'), mm=('mm', 'first')).reset_index(); g['r'] = (g.s + k * g.mm) / (g.n + k)
    r = d.merge(g[['m', 'p', 'r']], on=['m', 'p'], how='left').r.values.clip(1e-3, 1 - 1e-3)
    return np.log(r / (1 - r)).astype('float32')

CB_CAT = ['customer_id', 'product_id', 'store_id']
def make_X(part, te_part):
    return pd.concat([part[FEATS].reset_index(drop=True).astype('float32'), te_part.reset_index(drop=True)], axis=1)
def cb_X(part, X):
    return pd.concat([X, part[CB_CAT].reset_index(drop=True).astype(str).add_suffix('_cat')], axis=1)

LGB_PARAMS = dict(objective='binary', metric='auc', learning_rate=0.04, num_leaves=48, max_depth=8, min_child_samples=300, feature_fraction=0.7,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l1=2.0, lambda_l2=15.0, max_bin=255, verbose=-1, num_threads=-1, seed=SEED)
XGB_PARAMS = dict(objective='binary:logistic', eval_metric='auc', tree_method='hist', eta=0.05, max_depth=7, min_child_weight=30, subsample=0.8,
                  colsample_bytree=0.7, reg_lambda=15.0, reg_alpha=2.0, nthread=-1, seed=SEED)
CB_PARAMS = dict(learning_rate=0.08, depth=7, l2_leaf_reg=10, eval_metric='AUC', random_seed=SEED, thread_count=-1, verbose=300)
IN_CATS = [c for c in FEATS if c in CATS]

def fit_lgb(XA, yA, off, XB=None, yB=None, rounds=LGB_ROUNDS, seed=SEED):
    dA = lgb.Dataset(XA, yA, init_score=off, categorical_feature=IN_CATS, free_raw_data=False)
    p = dict(LGB_PARAMS, seed=seed)
    if XB is None: return lgb.train(p, dA, rounds)
    dB = lgb.Dataset(XB, yB, reference=dA, categorical_feature=IN_CATS)
    return lgb.train(p, dA, rounds, valid_sets=[dB], callbacks=[lgb.early_stopping(150, verbose=False)])

In [ ]:
def run_fold(v0, v1, mode, models=('lgb', 'xgb', 'cb', 'lr'), verbose=True):
    t0 = time.time()
    trm = (T.ts < v0).values; vam = ((T.ts >= v0) & (T.ts < v1)).values
    A, B = T[trm].reset_index(drop=True), T[vam].reset_index(drop=True); yA, yB = y_all[trm], y_all[vam]
    teA, teB = target_encode(A, yA, B); XA, XB = make_X(A, teA), make_X(B, teB)
    off = promo_offset(A, yA) if mode == 'offset' else None
    out, iters = {}, {}
    if 'lgb' in models:
        m = fit_lgb(XA, yA, off, XB, yB); out['lgb'] = m.predict(XB, num_iteration=m.best_iteration); iters['lgb'] = m.best_iteration
    if 'xgb' in models and USE_XGB:
        dA = xgb.DMatrix(XA, label=yA, base_margin=off); dB = xgb.DMatrix(XB, label=yB)
        m = xgb.train(XGB_PARAMS, dA, XGB_ROUNDS, evals=[(dB, 'val')], early_stopping_rounds=150, verbose_eval=False)
        out['xgb'] = m.predict(dB, iteration_range=(0, m.best_iteration + 1)); iters['xgb'] = m.best_iteration + 1
    if 'cb' in models and USE_CB:
        pa = Pool(cb_X(A, XA), yA, cat_features=[c + '_cat' for c in CB_CAT], baseline=off); pb = Pool(cb_X(B, XB), yB, cat_features=[c + '_cat' for c in CB_CAT])
        m = CatBoostClassifier(iterations=CB_ROUNDS, early_stopping_rounds=100, **CB_PARAMS); m.fit(pa, eval_set=pb, use_best_model=True)
        out['cb'] = m.predict_proba(pb)[:, 1]; iters['cb'] = m.get_best_iteration() + 1
    if 'lr' in models and USE_LR:
        out['lr'], _ = fit_lr(XA, yA, XB)
    if verbose: print(f'fold [{v0} .. {v1}] mode={mode} | ' + ' | '.join(f'{k}: {roc_auc_score(yB, v):.4f}' for k, v in out.items()) + f' | {time.time()-t0:.0f}s')
    return out, iters, yB

def fit_lr(XA, yA, XB, XT=None):
    """Standardised (quantile->normal) numerics + one-hot categoricals -> L2 logistic regression: a very different inductive bias."""
    num = [c for c in XA.columns if c not in CATS]; cat = [c for c in XA.columns if c in CATS]
    sub = np.random.RandomState(0).choice(len(XA), min(300000, len(XA)), replace=False)
    qt = QuantileTransformer(n_quantiles=200, output_distribution='normal', subsample=300000, random_state=0).fit(XA[num].iloc[sub].fillna(0))
    oh = OneHotEncoder(handle_unknown='ignore', min_frequency=50).fit(XA[cat])
    mk = lambda X: sparse.hstack([sparse.csr_matrix(qt.transform(X[num].fillna(0)).astype('float32')), oh.transform(X[cat])]).tocsr()
    lr = LogisticRegression(C=0.05, max_iter=300).fit(mk(XA), yA)
    return lr.predict_proba(mk(XB))[:, 1], (lr.predict_proba(mk(XT))[:, 1] if XT is not None else None)

### 5a. Choose how to neutralise the promo drift (LightGBM, latest fold + earlier fold)

In [ ]:
res = {}
for mode in ['drop', 'offset']:
    for i, (v0, v1) in enumerate(FOLDS):
        o, _, yB = run_fold(v0, v1, mode, models=('lgb',)); res[(mode, i)] = roc_auc_score(yB, o['lgb'])
# also show what happens if the promo code is naively used as a feature (what the old notebooks did)
v0, v1 = FOLDS[-1]; trm = (T.ts < v0).values; vam = ((T.ts >= v0) & (T.ts < v1)).values
A, B = T[trm].reset_index(drop=True), T[vam].reset_index(drop=True); teA, teB = target_encode(A, y_all[trm], B)
XA, XB = make_X(A, teA), make_X(B, teB); XA['promo'] = A.promo_channel_code.values; XB['promo'] = B.promo_channel_code.values
m = lgb.train(LGB_PARAMS, lgb.Dataset(XA, y_all[trm]), 400); naive = roc_auc_score(y_all[vam], m.predict(XB)); del A, B, XA, XB, m
print('\nnaive (promo as feature), last fold:', round(naive, 4)); print({k: round(v, 4) for k, v in res.items()})
MODE = 'offset' if res[('offset', len(FOLDS) - 1)] > res[('drop', len(FOLDS) - 1)] + 0.0005 else 'drop'
print('==> promo handling:', MODE)

### 5b. Time-based cross-validation of all models

In [ ]:
cv = []; preds_last = None
for i, (v0, v1) in enumerate(FOLDS):
    o, it, yB = run_fold(v0, v1, MODE); cv.append((o, it, yB))
names = list(cv[-1][0].keys())
cv_tbl = pd.DataFrame({n: [roc_auc_score(f[2], f[0][n]) for f in cv] for n in names}, index=[f'fold{i+1}' for i in range(len(cv))]); print(cv_tbl.round(4))
cv_tbl.T.plot.bar(figsize=(8, 3.5)); plt.ylim(cv_tbl.values.min() - .01, cv_tbl.values.max() + .01); plt.title('CV ROC-AUC by model / fold'); plt.show()

In [ ]:
# blend weights: rank average, simple grid on the latest fold (also reported on the earlier fold as a sanity check)
import itertools
rk = lambda p: rankdata(p) / len(p)
best_w, best_s = None, 0
grid = [w for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=len(names)) if abs(sum(w) - 1) < 1e-6]
R = [{n: rk(f[0][n]) for n in names} for f in cv]                                   # rank once, reuse for every grid point
score = lambda i, w: roc_auc_score(cv[i][2], sum(wi * R[i][n] for wi, n in zip(w, names)))
for w in grid:
    s = np.mean([score(i, w) for i in range(len(cv))])          # average over folds -> more stable than one fold
    if s > best_s: best_w, best_s = w, s
W = dict(zip(names, best_w)); print('blend weights:', {k: round(v, 2) for k, v in W.items()}, '| mean CV AUC: %.4f' % best_s)
print('per fold:', [round(score(i, best_w), 4) for i in range(len(cv))], '| best single (mean):', cv_tbl.mean().round(4).to_dict())

## 6. Final fit on all training data → predict test

In [ ]:
teF, teT = target_encode(T, y_all, TEST); XF, XT = make_X(T, teF), make_X(TEST, teT)
offF = promo_offset(T, y_all) if MODE == 'offset' else None
iters = {n: int(np.mean([f[1][n] for f in cv]) * 1.15) for n in names if n != 'lr'}
final = {}
if 'lgb' in names:
    ps = []
    for s in range(3):
        m = fit_lgb(XF, y_all, offF, rounds=iters['lgb'], seed=SEED + s); ps.append(rk(m.predict(XT)))
        if s == 0: imp = pd.Series(m.feature_importance('gain'), index=XF.columns).sort_values()
    final['lgb'] = np.mean(ps, axis=0); print('lgb done')
if 'xgb' in names:
    dF = xgb.DMatrix(XF, label=y_all, base_margin=offF); m = xgb.train(XGB_PARAMS, dF, iters['xgb']); final['xgb'] = rk(m.predict(xgb.DMatrix(XT))); print('xgb done')
if 'cb' in names:
    pf = Pool(cb_X(T, XF), y_all, cat_features=[c + '_cat' for c in CB_CAT], baseline=offF); pt = Pool(cb_X(TEST, XT), cat_features=[c + '_cat' for c in CB_CAT])
    m = CatBoostClassifier(iterations=iters['cb'], **CB_PARAMS); m.fit(pf); final['cb'] = rk(m.predict_proba(pt)[:, 1]); print('cb done')
if 'lr' in names:
    _, p = fit_lr(XF, y_all, XF.iloc[:5], XT); final['lr'] = rk(p); print('lr done')
blend = sum(W[n] * final[n] for n in names)
sub = pd.DataFrame({'order_line_id': TEST.order_line_id.values, 'returned': blend})
sub.to_csv('submission4.csv', index=False)
print(sub.shape, '| nulls:', int(sub.returned.isnull().sum()), '| unique ids:', sub.order_line_id.nunique(), '| range: %.3f-%.3f' % (blend.min(), blend.max()))
sub.head()

In [ ]:
# model-agreement check + feature importance
print(pd.DataFrame(final).corr(method='spearman').round(3))
imp.tail(30).plot.barh(figsize=(8, 9), color='darkorange'); plt.title('LightGBM feature importance (gain, top 30)'); plt.tight_layout(); plt.show()